# Deploying a machine learning model

Up to now, our models have lived inside notebooks. A notebook is a good place to explore, but nobody else can use a model that only exists in your Colab session. **Deployment** means making a model's predictions available to the people or systems that need them, reliably, safely, and repeatably.

In this notebook we take a house-price model (the Ames, Iowa housing data, predicted with the kNN regressor you met in Machine Learning I) from training to a service that anyone with a link can use. Every step uses open-source tools that work with any cloud or none:

| Step | Tool | Why this one |
| --- | --- | --- |
| Save the trained pipeline | `joblib` | Ships with scikit-learn |
| Validate inputs | `pydantic` | Rejects bad data before it reaches the model |
| Serve predictions over HTTP | `FastAPI` | The most common Python API framework; auto-generates docs |
| Test it | `pytest` | The standard Python test runner |
| Give users a screen | `Gradio` | A web UI in a few lines; free temporary public link from Colab |
| Package it | Docker | One image runs on a laptop, a university server, or any cloud |

By the end, you should be able to explain what changes when a model leaves the notebook, save and reload a pipeline safely, put a validated API in front of it, write tests that catch real failures, give it a user interface, and describe how to host and monitor it.

The code for the finished service lives in the [`app/`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/) folder next to this notebook. We walk through it piece by piece, and the `show()` helper prints the real source of each piece right here in the notebook. In Colab you can also browse the files: click the **folder icon** in the left sidebar and open `Kamilimu-2026-DS/Deploying ML Solutions/app/`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/deploying_ml_models.ipynb)

Click **Open in Colab** to run this notebook in Google Colab. Then run the next cell so the `app/` files load and the serving libraries are installed.

In [ ]:
import subprocess
import sys
from pathlib import Path

GITHUB_USER = "WanjiruCate"
GITHUB_REPO = "Kamilimu-2026-DS"
GITHUB_BRANCH = "main"
APP_DIR = "Deploying ML Solutions/app"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    dest = Path("/content") / GITHUB_REPO
    if not dest.exists():
        url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
        !git clone --depth 1 -b {GITHUB_BRANCH} {url} {dest}
    # Colab already has scikit-learn and pandas; add the serving libraries.
    !pip install -q fastapi httpx gradio
    import os
    os.chdir(dest / APP_DIR)
elif Path("app").exists():
    import os
    os.chdir("app")

sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

# show(x) prints the source code of a function or class from app/, so you can
# read the real implementation without leaving the notebook.
import inspect
from IPython.display import Code, display


def show(obj):
    display(Code(inspect.getsource(obj), language="python"))

## 1. What changes when a model leaves the notebook?

In a notebook, *you* are the only user. You know which cells to run, what the columns mean, and you notice when something looks odd. A deployed model has users who know none of that. So we must answer new questions:

- **Who uses it, and how?** A person filling in a form? Another program calling an API? A nightly job scoring a whole table?
- **What inputs will arrive?** Real users send missing fields, typos, negative areas, and values far outside anything the model has seen.
- **Which model is running?** When a prediction is questioned next month, can you say which version made it and on what data it was trained?
- **How do we know it still works?** After every code change, and as the world drifts away from the training data.

### Three common ways to deploy

| Pattern | How it works | Example |
| --- | --- | --- |
| **Batch** | Score many rows on a schedule; write results to a file or table | Score every new listing each night |
| **Real-time API** | A server answers one request at a time, in milliseconds | A website asks for a price as the user types |
| **Embedded / edge** | The model runs inside an app or device | An offline mobile app for field officers |

This notebook builds a real-time API, because it also supports a web UI and, in the next notebook, an AI agent that calls the model as a tool. Batch scoring is often simpler; choose it when nobody needs an answer immediately.

## 2. Train once, save the whole pipeline

In Machine Learning I you saw that kNN finds the *nearest* training houses by measuring distance between feature values. Distances are dominated by features with big numbers (living area in thousands of square feet) unless every feature is first put on the same scale with `StandardScaler`.

In a notebook, you scale the training data, fit the model, and scale the test data with the same scaler. In deployment, a new house arrives as raw numbers, and *someone has to remember* to apply that same scaler, with the same means and standard deviations learned from the training data, before calling the model. Forgetting it, or refitting the scaler on new data, is called **training-serving skew**: the model is served data prepared differently from the data it was trained on. It is one of the most common deployment bugs, because nothing crashes; the predictions are just wrong.

Let us see it happen. We train a scaler and a kNN model as two separate objects, then predict for a new house with and without scaling:

In [ ]:
import pandas as pd
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

data = pd.read_csv("../data/train.csv")
features = ["OverallQual", "GrLivArea", "GarageCars", "TotalBsmtSF", "YearBuilt", "FullBath"]
X, y = data[features], data["SalePrice"]

scaler = StandardScaler().fit(X)
knn = KNeighborsRegressor(n_neighbors=10).fit(scaler.transform(X), y)

new_house = pd.DataFrame([{"OverallQual": 7, "GrLivArea": 1710, "GarageCars": 2,
                           "TotalBsmtSF": 856, "YearBuilt": 2003, "FullBath": 2}])
print(f"Scaled like the training data: ${knn.predict(scaler.transform(new_house))[0]:,.0f}")
print(f"Forgot to scale:               ${knn.predict(new_house.to_numpy())[0]:,.0f}")

Same model, same house, a very different price, and no error message. The fix is a scikit-learn **`Pipeline`**: an object that bundles the steps (here: scale, then kNN) into one model. `pipeline.fit(X, y)` fits the scaler and then the model; `pipeline.predict(new_X)` applies the *saved* scaler and then the model. When we save the pipeline, the scaler is saved with it, so nobody can forget it at prediction time.

[`app/train.py`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/train.py) does three things:

1. builds a `Pipeline` (`StandardScaler` then `KNeighborsRegressor`) on six features;
2. records the validation MAE, and compares it with always predicting the median price and with linear regression, *before* refitting on all rows;
3. saves the pipeline with `joblib`, and a `metadata.json` describing it.

Here is the pipeline. Everything the model needs to turn raw columns into a prediction is inside it:

In [ ]:
import train

show(train.build_pipeline)

Run the training script:

In [ ]:
!{sys.executable} train.py

### Why save metadata?

A `.joblib` file on its own is an anonymous blob. The metadata turns it into something you can operate. Let us look at it:

In [ ]:
import json

metadata = json.loads(Path("model/metadata.json").read_text())
print(json.dumps(metadata, indent=2))

Each field answers a deployment question:

- `model_version` and `trained_at`: *which model made this prediction?* Every API response includes the version.
- `sklearn_version`: a pickled model should be loaded with the **same** scikit-learn version that saved it. Different versions may fail or, worse, silently behave differently. That is why [`app/requirements.txt`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/requirements.txt) pins versions.
- `numeric_ranges`: *what did the model see in training?* We use these to flag unusual inputs.
- `validation`: the evidence that the model is worth using. Its MAE is far lower than always predicting the median, and lower than linear regression on the same features.
- `intended_use`: what the model is, and is not, for.

> **Security note:** `joblib` and `pickle` files can run arbitrary code when loaded. Only load model files you created or fully trust. Never load a `.joblib` or `.pkl` file someone sent you, or one downloaded from an unknown source. (The `skops` library offers a safer format if you need to share models.)

## 3. Load the model and validate inputs

[`app/predictor.py`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/predictor.py) is the single place where the model is loaded and called. Both the API and the AI agent in the next notebook use it, so the rules are defined once.

The key idea is an **input contract**. `HouseFeatures` is a `pydantic` model that states which fields are required, their types, and allowed ranges. Anything that breaks the contract is rejected with a clear message *before* it reaches the model.

In [ ]:
import predictor

show(predictor.HouseFeatures)
show(predictor.predict)

In [ ]:
from pydantic import ValidationError
from predictor import HouseFeatures, predict

house = HouseFeatures(
    OverallQual=7, GrLivArea=1710, GarageCars=2, TotalBsmtSF=856,
    YearBuilt=2003, FullBath=2,
)
print(predict(house))

Now send the kinds of input real users send. Each one fails loudly instead of producing a confident but meaningless price:

In [ ]:
bad_inputs = [
    {"OverallQual": 12},              # outside the 1-10 scale
    {"GrLivArea": -300},              # impossible area
    {"GarageCars": 2.5},             # a garage holds whole cars
    {"YearBuilt": "recent"},          # wrong type
]
for change in bad_inputs:
    try:
        HouseFeatures(**{**house.model_dump(), **change})
    except ValidationError as exc:
        print(change, "->", exc.errors()[0]["msg"])

### Valid but unusual inputs

Some inputs are *possible* but outside what the model saw in training, such as an 8,000 sq ft house when the largest training house was about 5,600. kNN cannot extrapolate: its estimate is the average price of the 10 most similar training houses, so a house bigger than any it has seen simply gets the price of the biggest houses it knows. We should not refuse the request, but we should **warn**. This is the whole check:

In [ ]:
show(predictor.range_warnings)

In [ ]:
big_house = house.model_copy(update={"GrLivArea": 8000})
print(predict(big_house))

## 4. Put an API in front of the model

An **API** (application programming interface) lets other programs use your model over the web. A web app, a mobile app, a spreadsheet script, or an AI agent sends a request with the inputs as JSON, and gets back a JSON response with the prediction.

[`app/api.py`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/api.py) uses FastAPI. Look at how little code an endpoint needs:

```python
@app.post("/predict", response_model=Prediction)
def predict_price(features: HouseFeatures):
    return predict(features)
```

Because `features` is typed as `HouseFeatures`, FastAPI validates every request against our contract and returns a `422` error with details if it fails. The service also has:

| Endpoint | Purpose |
| --- | --- |
| `GET /health` | Lets the hosting platform check the service is alive |
| `GET /model-info` | Returns the metadata, so callers can see what they are using |
| `POST /predict` | One prediction |
| `POST /predict/batch` | Up to 100 predictions in one request |
| `POST /agent/chat` | The AI agent from the next notebook |
| `/docs` | Interactive documentation, generated automatically |
| `/ui` | The Gradio interface (section 6) |

We can call the API from the notebook without starting a server by using FastAPI's `TestClient`:

In [ ]:
!uv pip install gradio

In [ ]:
import api

show(api.health)
show(api.predict_price)

In [ ]:
from fastapi.testclient import TestClient
from api import app

client = TestClient(app)

print(client.get("/health").json())

response = client.post("/predict", json=house.model_dump())
print(response.status_code, response.json())

response = client.post("/predict", json={**house.model_dump(), "OverallQual": 12})
print(response.status_code, response.json()["detail"][0]["msg"])

### Running the real server on your laptop

From the `app/` folder in a terminal:

```bash
pip install -r requirements.txt
python train.py
uvicorn api:app --reload --port 8000
```

Open http://127.0.0.1:8000/docs, click **POST /predict → Try it out → Execute**. Or call it from another terminal:

```bash
curl -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"OverallQual": 7, "GrLivArea": 1710, "GarageCars": 2, "TotalBsmtSF": 856, "YearBuilt": 2003, "FullBath": 2}'
```

This is exactly what any frontend (React, Flutter, a PWA, a USSD gateway) would do.

## 5. Test before you deploy

A model that scored well in your notebook can still break in deployment because of a renamed column, a library upgrade, or a refactor. **Automated tests** are small programs that check the service still behaves correctly; run them after every change, and before every deployment.

Open [`app/tests/test_api.py`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/tests/test_api.py). It contains four kinds of test that are worth having in every ML project:

| Kind of test | Example in our file | What it catches |
| --- | --- | --- |
| **Smoke test** | `/health` returns `ok` | The service does not start |
| **Contract test** | Bad and missing inputs return `422` | Validation accidentally removed |
| **Behavioural test** | Higher quality is not cheaper, all else equal | A model that learned something absurd |
| **Sanity range** | A typical house costs between $50k and $500k | Wrong units, wrong target, broken preprocessing |

Behavioural tests are especially useful for ML: we cannot know the *exact* right price, but we know some relationships that must hold. Here is one, and the contract test: Run everything:

In [ ]:
from tests import test_api

show(test_api.test_better_quality_is_not_cheaper)
show(test_api.test_invalid_inputs_are_rejected)

[`app/tests/test_agent.py`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/tests/test_agent.py) tests the agent from the next notebook. Run every test:

In [ ]:
!{sys.executable} -m pytest -q

A deployment pipeline should refuse to deploy if any test fails. Our `Dockerfile` (section 7) runs the tests during the build, so a broken model never becomes a running service. On GitHub, you can run the same command on every push with a GitHub Actions workflow.

## 6. Give users a screen with Gradio

Most users will not send JSON. [`app/ui.py`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/ui.py) uses Gradio to build a form for the model and a chat window for the agent. It calls the same `predict()` function, so validation and warnings are identical.

Run the next cell. In Colab, Gradio prints a temporary public `gradio.live` link (valid for about a week) that you can send to a friend's phone. That is the quickest way to get user feedback on a prototype, but it disappears when your Colab session ends, so it is not a real deployment.

In [ ]:
from ui import build_ui

demo = build_ui()
demo.launch(share=IN_COLAB)

In [ ]:
# Stop the interface when you are done.
demo.close()

## 7. Package it with Docker, then host it anywhere

Your laptop has a particular Python version and set of libraries. The server you deploy to will have different ones. **Docker** solves this by packaging your code, its exact dependencies, and the command to start it into one **image**. Any machine with Docker runs the image the same way.

Here is our [`app/Dockerfile`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/Dockerfile), line by line:

```dockerfile
FROM python:3.12-slim                      # start from a small official Python image
RUN useradd --create-home appuser          # do not run as root
WORKDIR /home/appuser/app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt   # pinned versions
COPY --chown=appuser . .                   # our code and the saved model
USER appuser
ENV PORT=7860
RUN test -f model/house_price_model.joblib && python -m pytest -q   # refuse to build if tests fail
CMD uvicorn api:app --host 0.0.0.0 --port ${PORT}
```

Build and run it locally (Docker Desktop, Rancher Desktop, or Colima):

```bash
cd "Deploying ML Solutions/app"
docker build -t house-price-service .
docker run -p 7860:7860 house-price-service
# open http://localhost:7860/docs and http://localhost:7860/ui
```

Colab cannot run Docker, so do this part on your own machine or let the hosting platform build the image for you.

### Where to host it

Because the image is standard, you are not tied to any one company. Choose based on cost, where your users are, and data rules. Free tiers change often; check the current terms.

| Option | Free tier for students? | Notes |
| --- | --- | --- |
| **Hugging Face Spaces** (Docker SDK) | Yes, CPU | Push the `app/` folder to a Space; good for demos and portfolios |
| **Render**, **Railway**, **Fly.io** | Limited | Connect a GitHub repo; builds the Dockerfile on each push |
| **Google Cloud Run**, **Azure Container Apps**, **AWS App Runner** | Free monthly allowance (card required) | Scales to zero when idle; common in industry |
| **Your own server or VPS** | Pay per month | `docker run` behind a reverse proxy such as Caddy or Nginx; full control over where data lives |

### Example: deploy to a Hugging Face Space

1. Create a free account and a new Space. Choose **Docker** as the SDK and **Blank** as the template.
2. Clone the Space repository, copy everything inside `Deploying ML Solutions/app/` into it, and add this front matter to the top of its `README.md`:
   ```yaml
   ---
   title: House Price Service
   sdk: docker
   app_port: 7860
   ---
   ```
3. `git add . && git commit -m "Deploy service" && git push`. The Space builds the image, runs the tests, and starts the service.
4. If you later use a hosted LLM for the agent, add `AGENT_MODEL` and the provider key under **Settings → Variables and secrets**, never in the code.

The same folder deploys to Render or Cloud Run with no code changes; only the platform's setup steps differ.

## 8. Document it: README and model card

A deployment is not finished until someone else can understand, run, and question it. Two documents do most of the work:

- **README**: what the service does, how to install, train, test, and run it, the endpoints, and who to contact. See [`app/README.md`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/README.md).
- **Model card**: a short, honest description of the model: intended use and users, what it must *not* be used for, training data and its limits, evaluation results against a baseline, known failure cases, and how it is monitored. See [`app/MODEL_CARD.md`](https://github.com/WanjiruCate/Kamilimu-2026-DS/blob/main/Deploying%20ML%20Solutions/app/MODEL_CARD.md).

Write the model card for a reader who is deciding whether to trust the model. For example, our house-price model is trained on 2006-2010 sales from one US town, so it says nothing about house prices in Nairobi today. That sentence belongs in the model card.

## 9. After deployment: monitor and improve

A deployed model starts ageing immediately. The data it sees drifts away from the training data (new kinds of houses, inflation, a new type of user), and its accuracy quietly falls. Plan, from day one, to watch:

- **Service health:** uptime, error rate, response time. The `/health` endpoint and the host's logs cover the basics.
- **Input drift:** are inputs moving outside training ranges? Our range warnings are a simple version; count how often they fire.
- **Prediction drift:** is the distribution of predictions changing?
- **Real outcomes:** when true sale prices become known, compute the live MAE and compare it with the validation MAE in the metadata.
- **User feedback:** what do users say is wrong? The next notebook shows a simple feedback log.

When monitoring shows the model has degraded: retrain with newer data, bump `MODEL_VERSION`, run the tests, and redeploy. That loop is the start of **MLOps**.

Never log personal data you do not need. If your users are people (patients, students, customers), Kenya's Data Protection Act, 2019 applies to what you collect, store, and send to third-party services.

## 10. Exercises and career connections

1. Call `/predict` with three houses of your own design. Which inputs produce warnings? Why are warnings better than refusing the request?
2. Add a behavioural test: a house with more living area should not be cheaper, all else equal. Does it pass? If not, what does that tell you about the model?
3. Add a seventh feature to `train.py` (for example `LotArea`). What else must change so that the API, the UI, and the tests stay consistent? Bump the model version.
4. Write a `score_batch.py` script that reads `../data/test.csv`, calls `predict()` for each row, and writes `predictions.csv` with the model version in a column. When would batch scoring be a better choice than the API?
5. Deploy the service to a free host of your choice and share the `/ui` link with a classmate. Ask them to try to break it, and write down what they found.
6. **Innovation project:** for your own solution, write one paragraph each on: who calls the model and how (batch, API, or embedded), the input contract, two behavioural tests, where it will be hosted and why, and what you will monitor.

### Takeaways

- Save the whole preprocessing-plus-model pipeline, together with metadata about its version, training data, and evaluation.
- Validate inputs at the boundary. Reject impossible inputs and warn about unusual ones.
- An API makes the model usable by any frontend or agent; a UI makes it usable by people.
- Automated tests, including behavioural tests, should gate every deployment.
- A Docker image keeps you independent of any single cloud provider.
- Document the intended use and limits, and plan monitoring before launch.

### Exercise starter cells

You can do exercises 1, 2 and 4 entirely in the cells below. Exercise 3 needs edits to `train.py`, `predictor.py` and `ui.py`: in Colab, double-click a file in the folder sidebar to edit it, then **Runtime → Restart session** and re-run the notebook so Python loads your changes.

In [ ]:
# Exercise 1: design your own houses and call the API.
my_house = {**house.model_dump(), "OverallQual": 5, "GrLivArea": 1200, "YearBuilt": 1960}
print(client.post("/predict", json=my_house).json())

In [ ]:
# Exercise 2: a behavioural test for living area. Does it hold for houses of every age?
for year in [1920, 1960, 2000]:
    small = predict(house.model_copy(update={"GrLivArea": 1000, "YearBuilt": year}))
    large = predict(house.model_copy(update={"GrLivArea": 2500, "YearBuilt": year}))
    print(year, small.predicted_price, large.predicted_price, "OK" if large.predicted_price >= small.predicted_price else "FAILS")

In [ ]:
# Exercise 4: batch scoring. Complete the loop, then write predictions.csv.
import pandas as pd

scoring = pd.read_csv("../data/test.csv")
rows = []
for record in scoring[metadata["features"] + ["Id"]].to_dict("records"):
    # TODO: build HouseFeatures from `record` (some test rows have missing values:
    # decide what to do with them), call predict(), and append a dict with
    # Id, SalePrice and model_version to `rows`.
    pass

# pd.DataFrame(rows).to_csv("predictions.csv", index=False)